In [ ]:
# =============================================================================
# Cell 1: Install Dependencies
# =============================================================================
# We remove the numpy<2 restriction to avoid binary incompatibility with pandas/Python 3.12
!pip install -q numpy pandas scikit-learn torch xgboost rdkit \
    matplotlib seaborn scipy deepchem transformers

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 21.4 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# =============================================================================
# Cell 2: Configuration & Constants
# =============================================================================
import os, warnings
warnings.filterwarnings('ignore')

TOX21_TASKS = [
    'NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-ER', 'NR-ER-LBD',
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE',
    'SR-MMP', 'SR-p53', 'NR-Aromatase'
]

SEEDS = [42, 1, 7, 123, 999]

DATA_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"

MAX_ATOMS = 60
FP_NBITS  = 2048
FP_RADIUS = 2

# GNN hyperparameters
GNN_HIDDEN_DIM   = 128
GNN_DROPOUT      = 0.3
GNN_EPOCHS       = 30
GNN_LR           = 0.0005
GNN_WEIGHT_DECAY = 1e-4

# RF / XGBoost
RF_N_ESTIMATORS  = 200
XGB_N_ESTIMATORS = 200
XGB_LR           = 0.05
XGB_MAX_DEPTH    = 6

# DeepChem
DC_EPOCHS     = 20
DC_BATCH_SIZE = 64
DC_DROPOUT    = 0.2

RESULTS_DIR = "results"
FIGURES_DIR = "figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"Configuration loaded: {len(TOX21_TASKS)} endpoints, {len(SEEDS)} seeds")


Configuration loaded: 12 endpoints, 5 seeds


In [ ]:
import pandas as pd
import requests
import os

# Dosyayı indir ve yükle
print("Veri seti indiriliyor...")
df_raw = pd.read_csv(DATA_URL, compression='gzip')

print(f"Başarıyla yüklendi! Toplam satır sayısı: {len(df_raw)}")
display(df_raw.head())

Veri seti indiriliyor...
Başarıyla yüklendi! Toplam satır sayısı: 7831


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [ ]:
# =============================================================================
# Cell 3: Data Utilities — loading, fingerprints, graphs, scaffold split
# =============================================================================
import ssl, numpy as np, pandas as pd, torch
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold

# ── Data Loading ──────────────────────────────────────────────────────────────

def load_tox21_raw():
    """Load raw Tox21 dataset."""
    try:
        return pd.read_csv(DATA_URL, compression='gzip')
    except Exception:
        import urllib.request
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        os.makedirs('data', exist_ok=True)
        cache = 'data/tox21_full.csv.gz'
        opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ctx))
        with opener.open(DATA_URL) as resp, open(cache, 'wb') as out:
            out.write(resp.read())
        return pd.read_csv(cache, compression='gzip')


def load_tox21_task(df_raw, task):
    """Extract valid SMILES and labels for a single task."""
    df = df_raw[['smiles', task]].dropna()
    df.columns = ['SMILES', 'Label']
    valid = [Chem.MolFromSmiles(s) is not None for s in df['SMILES']]
    df = df[valid].reset_index(drop=True)
    return df['SMILES'].tolist(), df['Label'].values.astype(int)


def get_dataset_statistics(df_raw, tasks):
    """Per-endpoint dataset statistics (R2 minor comment)."""
    stats = []
    for task in tasks:
        n_total_raw = len(df_raw)
        df_task = df_raw[['smiles', task]].copy()
        n_labeled = df_task[task].notna().sum()
        n_missing = df_task[task].isna().sum()
        df_task = df_task.dropna()
        df_task.columns = ['SMILES', 'Label']
        valid_mask = [Chem.MolFromSmiles(s) is not None for s in df_task['SMILES']]
        df_valid = df_task[valid_mask].reset_index(drop=True)
        n_valid = len(df_valid)
        n_pos = int(df_valid['Label'].sum())
        pos_rate = n_pos / n_valid if n_valid > 0 else 0
        scaffolds = set()
        for s in df_valid['SMILES']:
            mol = Chem.MolFromSmiles(s)
            if mol:
                try:
                    sc = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
                    scaffolds.add(sc)
                except Exception:
                    pass
        stats.append({
            'Endpoint': task, 'N_total': n_total_raw, 'N_labeled': int(n_labeled),
            'N_missing': int(n_missing), 'N_valid': n_valid, 'N_positives': n_pos,
            'Positive_rate': round(pos_rate, 4), 'N_unique_scaffolds': len(scaffolds)
        })
    return pd.DataFrame(stats)

# ── Fingerprints ──────────────────────────────────────────────────────────────

def get_fingerprint(smiles, n_bits=FP_NBITS, radius=FP_RADIUS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def get_fingerprints_batch(smiles_list):
    return np.stack([get_fingerprint(s) for s in smiles_list])

def get_morgan_fp_objects(smiles_list, n_bits=FP_NBITS, radius=FP_RADIUS):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits))
        else:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(
                Chem.MolFromSmiles('C'), radius, nBits=n_bits))
    return fps

# ── Graph Construction ────────────────────────────────────────────────────────

ATOM_SYMBOLS = [
    'C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca','Fe','As','Al','I',
    'B','V','K','Tl','Yb','Sb','Sn','Ag','Pd','Co','Se','Ti','Zn','H','Li',
    'Ge','Cu','Au','Ni','Cd','In','Mn','Zr','Cr','Pt','Hg','Pb','Unknown'
]
HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2, 'Other'
]
CHIRALITY_TYPES = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW, 'Other'
]

def _one_hot(x, allowable_set):
    if x not in allowable_set:
        x = allowable_set[-1]
    return [x == s for s in allowable_set]

def atom_features(atom, include_chirality=False):
    features = (
        _one_hot(atom.GetSymbol(), ATOM_SYMBOLS) +
        _one_hot(atom.GetTotalDegree(), list(range(11))) +
        _one_hot(atom.GetTotalNumHs(), list(range(5))) +
        _one_hot(atom.GetImplicitValence(), list(range(6))) +
        _one_hot(atom.GetHybridization(), HYBRIDIZATIONS) +
        [atom.GetIsAromatic(), atom.IsInRing()]
    )
    if include_chirality:
        features += _one_hot(atom.GetChiralTag(), CHIRALITY_TYPES)
    return features

def get_atom_feature_dim(include_chirality=False):
    dim = 44 + 11 + 5 + 6 + 6 + 2  # = 74
    if include_chirality:
        dim += 4
    return dim

def smiles_to_graph(smiles, max_atoms=MAX_ATOMS, include_chirality=False):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() > max_atoms:
        return None
    num_atoms = mol.GetNumAtoms()
    feats = np.array([atom_features(a, include_chirality) for a in mol.GetAtoms()], dtype=np.float32)
    pad_len = max_atoms - num_atoms
    feats_padded = np.pad(feats, ((0, pad_len), (0, 0)), mode='constant')
    adj = np.eye(max_atoms, dtype=np.float32)
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        adj[i, j] = adj[j, i] = 1.0
    mask = np.zeros(max_atoms, dtype=np.float32)
    mask[:num_atoms] = 1.0
    return {
        'adj': torch.tensor(adj), 'feat': torch.tensor(feats_padded),
        'mask': torch.tensor(mask).unsqueeze(1)
    }

# ── Scaffold Splitting ───────────────────────────────────────────────────────

def generate_scaffolds(smiles_list, include_chirality=False):
    scaffolds = {}
    for idx, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        sc = ""
        if mol:
            try:
                sc = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=include_chirality)
            except Exception:
                pass
        scaffolds.setdefault(sc, []).append(idx)
    return sorted(scaffolds.values(), key=len, reverse=True)

def scaffold_split(smiles_list, train_ratio=0.64, val_ratio=0.16, include_chirality=False):
    groups = generate_scaffolds(smiles_list, include_chirality)
    train_i, val_i, test_i = [], [], []
    train_cut = len(smiles_list) * train_ratio
    val_cut   = len(smiles_list) * (train_ratio + val_ratio)
    cnt = 0
    for g in groups:
        if cnt < train_cut:      train_i.extend(g)
        elif cnt < val_cut:      val_i.extend(g)
        else:                    test_i.extend(g)
        cnt += len(g)
    return train_i, val_i, test_i

def scaffold_split_with_variance(smiles_list, n_partitions=3, include_chirality=False):
    scaffold_sets = generate_scaffolds(smiles_list, include_chirality)
    all_splits = []
    for p_seed in range(n_partitions):
        rng = np.random.RandomState(p_seed * 1000 + 7)
        size_groups = {}
        for g in scaffold_sets:
            size_groups.setdefault(len(g), []).append(g)
        ordered = []
        for sz in sorted(size_groups.keys(), reverse=True):
            gs = size_groups[sz]
            idxs = list(range(len(gs)))
            rng.shuffle(idxs)
            for i in idxs:
                ordered.append(gs[i])
        tr, va, te = [], [], []
        tr_cut = len(smiles_list) * 0.64
        va_cut = len(smiles_list) * 0.80
        cnt = 0
        for g in ordered:
            if cnt < tr_cut:   tr.extend(g)
            elif cnt < va_cut: va.extend(g)
            else:              te.extend(g)
            cnt += len(g)
        all_splits.append((tr, va, te))
    return all_splits

# ── Tanimoto Similarity ──────────────────────────────────────────────────────

def compute_nn_tanimoto(test_smiles, train_smiles):
    train_fps = get_morgan_fp_objects(train_smiles)
    test_fps  = get_morgan_fp_objects(test_smiles)
    nn_sims = []
    for tfp in test_fps:
        sims = DataStructs.BulkTanimotoSimilarity(tfp, train_fps)
        nn_sims.append(max(sims) if sims else 0.0)
    return np.array(nn_sims)

def bin_by_similarity(nn_sims):
    labels = []
    for s in nn_sims:
        if s < 0.3:   labels.append('low')
        elif s < 0.5: labels.append('medium')
        else:         labels.append('high')
    return np.array(labels)

print("Data utilities loaded.")


Data utilities loaded.


In [ ]:
# =============================================================================
# Cell 4: Model Definitions — RF, XGB, GCN, GIN, DeepChem, ChemBERTa
# =============================================================================
import torch, torch.nn as nn, torch.optim as optim
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Classical ML ──────────────────────────────────────────────────────────────

def create_rf(seed=42):
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS, class_weight='balanced',
        random_state=seed, n_jobs=-1)

def create_xgb(scale_pos_weight=1.0, seed=42):
    return XGBClassifier(
        n_estimators=XGB_N_ESTIMATORS, learning_rate=XGB_LR,
        max_depth=XGB_MAX_DEPTH, scale_pos_weight=scale_pos_weight,
        random_state=seed, n_jobs=-1, eval_metric='logloss', verbosity=0)

# ── AdvancedGCN ───────────────────────────────────────────────────────────────

class AdvancedGCN(nn.Module):
    def __init__(self, n_features, hidden_dim=GNN_HIDDEN_DIM, dropout=GNN_DROPOUT):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(n_features, hidden_dim) * 0.01)
        self.W2 = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.W3 = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, adj, x, mask):
        h = self.relu(torch.mm(torch.mm(adj, x), self.W1)); h = self.dropout(h)
        h = self.relu(torch.mm(torch.mm(adj, h), self.W2)); h = self.dropout(h)
        h = self.relu(torch.mm(torch.mm(adj, h), self.W3))
        h = h * mask
        emb = torch.sum(h, dim=0) / (torch.sum(mask, dim=0) + 1e-6)
        return torch.sigmoid(self.fc2(self.relu(self.fc1(emb))))

# ── GIN ───────────────────────────────────────────────────────────────────────

class GINLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(nn.Linear(in_dim, out_dim), nn.ReLU(),
                                 nn.Linear(out_dim, out_dim))
    def forward(self, adj, h):
        neighbor_sum = torch.mm(adj, h) - h
        return self.mlp((1 + self.eps) * h + neighbor_sum)

class GINModel(nn.Module):
    def __init__(self, n_features, hidden_dim=128, n_layers=3, dropout=0.3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.layers.append(GINLayer(n_features, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(n_layers - 1):
            self.layers.append(GINLayer(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, adj, x, mask):
        h = x
        for layer, bn in zip(self.layers, self.bns):
            h = self.dropout(self.relu(bn(layer(adj, h))))
        h = h * mask
        emb = torch.sum(h, dim=0) / (torch.sum(mask, dim=0) + 1e-6)
        return torch.sigmoid(self.fc2(self.relu(self.fc1(emb))))

# ── GNN Train / Predict ──────────────────────────────────────────────────────

def train_gcn(model, train_graphs, train_labels, device,
              epochs=GNN_EPOCHS, lr=GNN_LR, weight_decay=GNN_WEIGHT_DECAY):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(train_graphs))
        for i in perm:
            g = train_graphs[i]
            if g is None: continue
            optimizer.zero_grad()
            pred = model(g['adj'].to(device), g['feat'].to(device), g['mask'].to(device))
            loss = nn.BCELoss()(pred.view(-1),
                                torch.tensor([train_labels[i]], dtype=torch.float32).to(device))
            loss.backward(); optimizer.step()
    return model

def predict_gcn(model, graphs, device, default_prob=0.5):
    model.eval()
    preds = []
    for g in graphs:
        if g is None: preds.append(default_prob); continue
        with torch.no_grad():
            p = model(g['adj'].to(device), g['feat'].to(device), g['mask'].to(device)).item()
        preds.append(p)
    return np.array(preds)

# ── DeepChem Wrapper ──────────────────────────────────────────────────────────

def run_deepchem_model(model_name, smiles_list, labels, train_idx, val_idx,
                       test_idx, seed=42):
    import deepchem as dc, tensorflow as tf
    np.random.seed(seed); tf.random.set_seed(seed)
    if model_name in ('AttentiveFP', 'MPNN'):
        featurizer = dc.feat.MolGraphConvFeaturizer(use_edges=True)
    else:
        featurizer = dc.feat.ConvMolFeaturizer()
    X = featurizer.featurize(smiles_list)
    def _valid(idxs):
        return [i for i in idxs if hasattr(X[i], 'atom_features') or hasattr(X[i], 'node_features')]
    vi_tr, vi_va, vi_te = _valid(train_idx), _valid(val_idx), _valid(test_idx)
    y = labels.reshape(-1, 1).astype(np.float32)
    mk = lambda idxs: dc.data.NumpyDataset(np.array([X[i] for i in idxs]), y[idxs])
    train_ds, val_ds, test_ds = mk(vi_tr), mk(vi_va), mk(vi_te)
    builders = {
        'AttentiveFP': lambda: dc.models.AttentiveFPModel(n_tasks=1, mode='classification',
                         dropout=DC_DROPOUT, batch_size=DC_BATCH_SIZE, random_seed=seed),
        'MPNN': lambda: dc.models.MPNNModel(n_tasks=1, mode='classification',
                         dropout=DC_DROPOUT, batch_size=DC_BATCH_SIZE, random_seed=seed),
        'GraphConv': lambda: dc.models.GraphConvModel(n_tasks=1, mode='classification',
                         dropout=DC_DROPOUT, batch_size=DC_BATCH_SIZE, random_seed=seed),
    }
    model = builders[model_name]()
    model.fit(train_ds, nb_epoch=DC_EPOCHS)
    def _extract(preds):
        if preds.ndim == 3: return preds[:, 0, 1]
        elif preds.ndim == 2: return preds[:, 1] if preds.shape[1] == 2 else preds[:, 0]
        return preds.flatten()
    val_probs  = _extract(model.predict(val_ds))
    test_probs = _extract(model.predict(test_ds))
    return val_probs, test_probs, vi_va, vi_te

# ── ChemBERTa ────────────────────────────────────────────────────────────────

def run_chemberta(smiles_list, labels, train_idx, val_idx, test_idx, seed=42,
                  model_name='seyonec/ChemBERTa-zinc-base-v1', epochs=5,
                  batch_size=32, lr=2e-5):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
    from torch.utils.data import Dataset
    import torch.nn.functional as F, shutil
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    class SmDS(Dataset):
        def __init__(self, smi, lab, tok, ml=128):
            self.enc = tok(smi, truncation=True, padding='max_length', max_length=ml, return_tensors='pt')
            self.lab = torch.tensor(lab, dtype=torch.long)
        def __len__(self): return len(self.lab)
        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}; item['labels'] = self.lab[i]; return item
    tr_ds = SmDS([smiles_list[i] for i in train_idx], labels[train_idx], tokenizer)
    va_ds = SmDS([smiles_list[i] for i in val_idx],   labels[val_idx],   tokenizer)
    te_ds = SmDS([smiles_list[i] for i in test_idx],  labels[test_idx],  tokenizer)
    args = TrainingArguments(
        output_dir=f'./chemberta_tmp_{seed}', num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
        learning_rate=lr, seed=seed, logging_steps=500, save_strategy='no',
        report_to='none', disable_tqdm=True)
    trainer = Trainer(model=base_model, args=args, train_dataset=tr_ds, eval_dataset=va_ds)
    trainer.train()
    def _probs(ds):
        logits = torch.tensor(trainer.predict(ds).predictions)
        return F.softmax(logits, dim=-1)[:, 1].numpy()
    vp, tp = _probs(va_ds), _probs(te_ds)
    shutil.rmtree(f'./chemberta_tmp_{seed}', ignore_errors=True)
    return vp, tp

print("All model definitions loaded.")


All model definitions loaded.


In [ ]:
# =============================================================================
# Cell 5: Meta-Learner Variants (R1.2, R2.5)
# =============================================================================
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize

def create_meta_learner(variant='lr_none'):
    if variant == 'lr_none':
        return LogisticRegression(C=np.inf, solver='lbfgs', max_iter=1000)
    elif variant == 'lr_l2':
        return LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000)
    elif variant == 'lr_elasticnet':
        return LogisticRegression(penalty='elasticnet', C=1.0, l1_ratio=0.5,
                                  solver='saga', max_iter=2000)
    elif variant in ('averaging', 'voting', 'constrained'):
        return None
    else:
        raise ValueError(f"Unknown variant: {variant}")

class ConstrainedMetaLearner:
    def __init__(self): self.weights = None
    def fit(self, X, y):
        n = X.shape[1]
        def loss(w):
            p = np.clip(X @ w, 1e-7, 1 - 1e-7)
            return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        res = minimize(loss, np.ones(n)/n, method='SLSQP',
                       constraints=[{'type':'eq','fun':lambda w: np.sum(w)-1}],
                       bounds=[(0,1)]*n)
        self.weights = res.x; return self
    def predict_proba(self, X):
        p = np.clip(X @ self.weights, 0, 1)
        return np.column_stack([1-p, p])

META_LEARNER_VARIANTS = ['lr_none', 'lr_l2', 'lr_elasticnet',
                         'averaging', 'voting', 'constrained']

def meta_predict(expert_probs, y_val=None, variant='lr_none', expert_probs_val=None):
    names = sorted(expert_probs.keys())
    X_test = np.column_stack([expert_probs[n] for n in names])
    info = {'variant': variant, 'expert_names': names}
    if variant == 'averaging':
        info['weights'] = np.ones(len(names))/len(names)
        return X_test.mean(axis=1), info
    if variant == 'voting':
        info['weights'] = np.ones(len(names))/len(names)
        return (X_test > 0.5).astype(float).mean(axis=1), info
    X_val = np.column_stack([expert_probs_val[n] for n in names])
    if variant == 'constrained':
        m = ConstrainedMetaLearner().fit(X_val, y_val)
        info['weights'] = m.weights; info['intercept'] = 0.0
        return m.predict_proba(X_test)[:, 1], info
    m = create_meta_learner(variant)
    m.fit(X_val, y_val)
    info['weights'] = m.coef_[0]; info['intercept'] = m.intercept_[0]
    return m.predict_proba(X_test)[:, 1], info

def override_analysis(expert_probs, hybrid_probs, y_true, expert_names):
    results = {}
    for name in expert_names:
        ep = (expert_probs[name] > 0.5).astype(int)
        hp = (hybrid_probs > 0.5).astype(int)
        ovr = ep != hp
        n_ovr = ovr.sum()
        if n_ovr > 0:
            ens_correct = (hp[ovr] == y_true[ovr]).sum()
            exp_correct = (ep[ovr] == y_true[ovr]).sum()
            rate = ens_correct / n_ovr
        else:
            ens_correct, exp_correct, rate = 0, 0, np.nan
        results[name] = {
            'n_overrides': int(n_ovr), 'override_rate': n_ovr/len(y_true),
            'ensemble_correct': int(ens_correct), 'expert_correct': int(exp_correct),
            'override_success_rate': rate
        }
    return results

print("Meta-learner variants loaded.")


Meta-learner variants loaded.


In [ ]:
# =============================================================================
# Cell 6: Metrics — ROC-AUC, PR-AUC, MCC, Balanced Acc, Calibration, ECE (R2.3)
# =============================================================================
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             matthews_corrcoef, balanced_accuracy_score)
from sklearn.calibration import calibration_curve
from scipy import stats

def compute_all_metrics(y_true, y_prob, threshold=0.5):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    m = {}
    try: m['ROC_AUC'] = roc_auc_score(y_true, y_prob)
    except: m['ROC_AUC'] = np.nan
    try: m['PR_AUC'] = average_precision_score(y_true, y_prob)
    except: m['PR_AUC'] = np.nan
    try: m['MCC'] = matthews_corrcoef(y_true, y_pred)
    except: m['MCC'] = np.nan
    try: m['Balanced_Acc'] = balanced_accuracy_score(y_true, y_pred)
    except: m['Balanced_Acc'] = np.nan
    return m

def compute_calibration(y_true, y_prob, n_bins=10):
    try:
        return calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')
    except:
        return np.array([]), np.array([])

def compute_ece(y_true, y_prob, n_bins=10):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= edges[i]) & (y_prob < edges[i+1])
        if mask.sum() == 0: continue
        ece += mask.sum() / len(y_true) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

def selective_prediction_metrics(y_true, y_prob,
                                  thresholds=[0.5, 0.6, 0.7, 0.8, 0.9]):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    confidence = np.maximum(y_prob, 1 - y_prob)
    results = []
    for t in thresholds:
        mask = confidence >= t
        cov = mask.sum() / len(y_true)
        if mask.sum() < 5:
            results.append({'threshold': t, 'coverage': cov,
                            'ROC_AUC': np.nan, 'PR_AUC': np.nan}); continue
        try: roc = roc_auc_score(y_true[mask], y_prob[mask])
        except: roc = np.nan
        try: pr = average_precision_score(y_true[mask], y_prob[mask])
        except: pr = np.nan
        results.append({'threshold': t, 'coverage': cov, 'ROC_AUC': roc, 'PR_AUC': pr})
    return pd.DataFrame(results)

# ── Statistical Tests (R2.8) ─────────────────────────────────────────────────

def tost_equivalence(x, y, delta):
    diff = np.asarray(x) - np.asarray(y)
    n = len(diff); mu = np.mean(diff); se = np.std(diff, ddof=1) / np.sqrt(n); df = n - 1
    p1 = 1 - stats.t.cdf((mu - (-delta)) / se, df=df)
    p2 = stats.t.cdf((mu - delta) / se, df=df)
    t_crit = stats.t.ppf(0.95, df=df)
    return max(p1, p2), mu, mu - t_crit*se, mu + t_crit*se

def cohens_d(x, y):
    d = np.asarray(x) - np.asarray(y)
    return np.mean(d) / np.std(d, ddof=1) if np.std(d, ddof=1) > 0 else 0.0

def bootstrap_ci(x, y, n_boot=10000, ci=0.95, seed=42):
    rng = np.random.RandomState(seed)
    d = np.asarray(x) - np.asarray(y)
    bm = np.array([np.mean(rng.choice(d, len(d), replace=True)) for _ in range(n_boot)])
    alpha = (1-ci)/2
    return np.mean(d), np.percentile(bm, alpha*100), np.percentile(bm, (1-alpha)*100)

print("Metrics and statistical tests loaded.")


Metrics and statistical tests loaded.


In [ ]:
# =============================================================================
# Cell 7 / Phase 1: Dataset Statistics (R2 minor)
# =============================================================================
print("=" * 80)
print("PHASE 1: Per-Endpoint Dataset Statistics")
print("=" * 80)

df_raw = load_tox21_raw()
print(f"Loaded Tox21 dataset: {df_raw.shape[0]} compounds, {df_raw.shape[1]} columns\n")

stats_df = get_dataset_statistics(df_raw, TOX21_TASKS)
stats_df.to_csv(f'{RESULTS_DIR}/dataset_statistics.csv', index=False)

from IPython.display import display
display(stats_df.style.set_caption("Table S1: Per-Endpoint Dataset Characteristics")
        .format({'Positive_rate': '{:.2%}'}))


PHASE 1: Per-Endpoint Dataset Statistics
Loaded Tox21 dataset: 7831 compounds, 14 columns



[00:46:36] WARNING: not removing hydrogen atom without neighbors
[00:46:37] Explicit valence for atom # 3 Al, 6, is greater than permitted
[00:46:37] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:46:37] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:46:37] Explicit valence for atom # 9 Al, 6, is greater than permitted
[00:46:37] Explicit valence for atom # 5 Al, 6, is greater than permitted
[00:46:37] Explicit valence for atom # 16 Al, 6, is greater than permitted
[00:46:38] Explicit valence for atom # 20 Al, 6, is greater than permitted
[00:46:38] WARNING: not removing hydrogen atom without neighbors
[00:46:45] WARNING: not removing hydrogen atom without neighbors
[00:46:45] Explicit valence for atom # 3 Al, 6, is greater than permitted
[00:46:45] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:46:46] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:46:46] Explicit valence for atom # 9 Al, 6, is greater than

,Endpoint,N_total,N_labeled,N_missing,N_valid,N_positives,Positive_rate,N_unique_scaffolds
0,NR-AhR,7831,6549,1282,6542,768,11.74%,1926
1,NR-AR,7831,7265,566,7258,308,4.24%,2159
2,NR-AR-LBD,7831,6758,1073,6751,237,3.51%,1946
3,NR-ER,7831,6193,1638,6186,791,12.79%,1769
4,NR-ER-LBD,7831,6955,876,6948,349,5.02%,2025
5,NR-PPAR-gamma,7831,6450,1381,6443,186,2.89%,1812
6,SR-ARE,7831,5832,1999,5825,942,16.17%,1576
7,SR-ATAD5,7831,7072,759,7065,264,3.74%,2049
8,SR-HSE,7831,6467,1364,6460,372,5.76%,1815
9,SR-MMP,7831,5810,2021,5804,918,15.82%,1668


In [ ]:
!pip install -q xgboost rdkit transformers torch scikit-learn pandas matplotlib seaborn scipy

In [ ]:
# ==============================================================================
# Cell 8:  MODELS — Pure PyTorch
# ==============================================================================
# Runs GraphConv, AttentiveFP, MPNN, GIN using PyTorch on GPU.
# ==============================================================================

import os, sys, functools, gc, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             matthews_corrcoef, balanced_accuracy_score)

RDLogger.DisableLog('rdApp.*')
print = functools.partial(print, flush=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("=" * 60)
print(f"Cell 8b: Fix Failed Models — PyTorch on {device}")
print("=" * 60)
if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ==============================================================================
# STEP 1: Load existing results and identify what needs re-running
# ==============================================================================
RESULTS_DIR = '/content/drive/MyDrive/tox-report-paper'
checkpoint_file = f'{RESULTS_DIR}/revised_full_benchmark.csv'

df_existing = pd.read_csv(checkpoint_file)
print(f"Loaded {len(df_existing)} existing rows")
print(f"Models: {sorted(df_existing['Model'].unique())}")

nan_rows = df_existing[df_existing['ROC_AUC'].isna()]
print(f"\nNaN rows: {len(nan_rows)}")
if len(nan_rows) > 0:
    print(nan_rows.groupby('Model').size())

MODELS_TO_FIX = ['AttentiveFP', 'GraphConv', 'MPNN', 'GIN']
print(f"\nWill re-run: {MODELS_TO_FIX}")

df_keep = df_existing[~df_existing['Model'].isin(MODELS_TO_FIX)].copy()
print(f"Keeping {len(df_keep)} good rows (RF, XGB, GCN, ChemBERTa, Hybrid_MoE)")

# ==============================================================================
# STEP 2: Data loading and splitting
# ==============================================================================
TOX21_TASKS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER',
    'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE',
    'SR-MMP', 'SR-p53'
]
SEEDS = [42, 1, 7, 123, 999]

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df_raw = pd.read_csv(url, compression='gzip')

def generate_scaffolds(dataset):
    scaffolds = {}
    for idx, smiles in enumerate(dataset):
        mol = Chem.MolFromSmiles(smiles)
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False) if mol else ""
        if scaffold not in scaffolds: scaffolds[scaffold] = []
        scaffolds[scaffold].append(idx)
    return sorted(scaffolds.values(), key=lambda x: len(x), reverse=True)

def scaffold_split(smiles_list):
    scaffold_sets = generate_scaffolds(smiles_list)
    train_idxs, val_idxs, test_idxs = [], [], []
    train_cutoff = len(smiles_list) * 0.64
    val_cutoff = len(smiles_list) * 0.80
    current_count = 0
    for group in scaffold_sets:
        if current_count < train_cutoff: train_idxs.extend(group)
        elif current_count < val_cutoff: val_idxs.extend(group)
        else: test_idxs.extend(group)
        current_count += len(group)
    return train_idxs, val_idxs, test_idxs

def load_tox21_task(df, task):
    df_task = df[['smiles', task]].dropna()
    mols = [Chem.MolFromSmiles(s) for s in df_task['smiles']]
    valid = [m is not None for m in mols]
    df_task = df_task[valid].reset_index(drop=True)
    return df_task['smiles'].tolist(), df_task[task].values.astype(int)

def compute_all_metrics(y_true, y_prob):
    y_prob = np.clip(np.nan_to_num(y_prob, nan=0.5), 0, 1)
    try: roc = roc_auc_score(y_true, y_prob)
    except: roc = np.nan
    try: pr = average_precision_score(y_true, y_prob)
    except: pr = np.nan
    y_pred = (y_prob >= 0.5).astype(int)
    try: mcc = matthews_corrcoef(y_true, y_pred)
    except: mcc = np.nan
    try: bacc = balanced_accuracy_score(y_true, y_pred)
    except: bacc = np.nan
    return {'ROC_AUC': roc, 'PR_AUC': pr, 'MCC': mcc, 'Balanced_Acc': bacc}

# ==============================================================================
# STEP 3: Graph construction (shared by all models)
# ==============================================================================
MAX_ATOMS = 60
SYMBOLS = ['C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca','Fe','As',
           'Al','I','B','V','K','Tl','Yb','Sb','Sn','Ag','Pd','Co','Se',
           'Ti','Zn','H','Li','Ge','Cu','Au','Ni','Cd','In','Mn','Zr','Cr',
           'Pt','Hg','Pb','Unknown']

HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
    'Other'
]

def one_hot(x, allowable_set):
    if x not in allowable_set: x = allowable_set[-1]
    return [x == s for s in allowable_set]

def smiles_to_graph(smiles, max_atoms=MAX_ATOMS):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    num_atoms = mol.GetNumAtoms()
    if num_atoms > max_atoms: return None
    features = []
    for atom in mol.GetAtoms():
        features.append(
            one_hot(atom.GetSymbol(), SYMBOLS) +          # 44
            one_hot(atom.GetTotalDegree(), list(range(11))) +  # 11
            one_hot(atom.GetTotalNumHs(), list(range(5))) +    # 5
            one_hot(atom.GetImplicitValence(), list(range(6))) + # 6
            one_hot(atom.GetHybridization(), HYBRIDIZATIONS) +   # 6
            [atom.GetIsAromatic(), atom.IsInRing()])            # 2
    features = np.array(features, dtype=np.float32)
    pad_len = max_atoms - num_atoms
    features_padded = np.pad(features, ((0, pad_len), (0, 0)), mode='constant')
    adj = np.eye(max_atoms, dtype=np.float32)
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        adj[i, j] = adj[j, i] = 1.0
    mask = np.zeros(max_atoms, dtype=np.float32)
    mask[:num_atoms] = 1.0
    return {
        'adj': torch.tensor(adj),
        'feat': torch.tensor(features_padded),
        'mask': torch.tensor(mask).unsqueeze(1)
    }

N_FEAT = 74  # 44 (symbol) + 11 (degree) + 5 (Hs) + 6 (valence) + 6 (hybridization) + 2 (aromatic, ring)

# ==============================================================================
# STEP 4: Model Definitions — All Pure PyTorch
# ==============================================================================

# ── GIN (Graph Isomorphism Network) ──────────────────────────────────────────
# Matches src/models.py GINModel: self-loop subtraction, BatchNorm, same interface as GCN
class GINLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.ReLU(),
            nn.Linear(out_dim, out_dim))

    def forward(self, adj, h):
        # adj includes self-loops; subtract h to avoid double-counting
        neighbor_sum = torch.mm(adj, h) - h
        out = (1 + self.eps) * h + neighbor_sum
        return self.mlp(out)

class GINModel(nn.Module):
    def __init__(self, n_features, hidden_dim=128, n_layers=3, dropout=0.3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.layers.append(GINLayer(n_features, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(n_layers - 1):
            self.layers.append(GINLayer(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, adj, x, mask):
        h = x
        for layer, bn in zip(self.layers, self.bns):
            h = layer(adj, h)
            h = bn(h)
            h = self.relu(h)
            h = self.dropout(h)
        h = h * mask
        emb = torch.sum(h, dim=0) / (torch.sum(mask, dim=0) + 1e-6)
        out = self.relu(self.fc1(emb))
        return torch.sigmoid(self.fc2(out))

# ── GraphConv (GCN — Kipf & Welling style) ──────────────────────────────────
# Matches src/models.py AdvancedGCN: A @ X @ W pattern with dropout
class GraphConvModel(nn.Module):
    def __init__(self, n_features, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(n_features, hidden_dim) * 0.01)
        self.W2 = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.W3 = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, adj, x, mask):
        h = self.relu(torch.mm(torch.mm(adj, x), self.W1))
        h = self.dropout(h)
        h = self.relu(torch.mm(torch.mm(adj, h), self.W2))
        h = self.dropout(h)
        h = self.relu(torch.mm(torch.mm(adj, h), self.W3))
        h = h * mask
        emb = torch.sum(h, dim=0) / (torch.sum(mask, dim=0) + 1e-6)
        out = self.relu(self.fc1(emb))
        return torch.sigmoid(self.fc2(out))

# ── AttentiveFP (Graph Attention Network) ────────────────────────────────────
class AttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4):
        super().__init__()
        self.heads = heads
        self.out_dim = out_dim
        self.W = nn.Linear(in_dim, out_dim * heads, bias=False)
        self.a = nn.Parameter(torch.randn(heads, 2 * out_dim))
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, adj, x, mask):
        N = x.shape[0]
        h = self.W(x).view(N, self.heads, self.out_dim)
        hi = h.unsqueeze(1).expand(-1, N, -1, -1)
        hj = h.unsqueeze(0).expand(N, -1, -1, -1)
        cat = torch.cat([hi, hj], dim=-1)
        e = self.leaky((cat * self.a).sum(-1))
        edge_mask = adj.unsqueeze(-1)
        e = e.masked_fill(edge_mask == 0, -1e9)
        alpha = torch.softmax(e, dim=1)
        out = torch.einsum('ijh,jhd->ihd', alpha, h).mean(dim=1)
        return out * mask

class AttentiveFPModel(nn.Module):
    def __init__(self, n_features, hidden_dim=64, heads=4):
        super().__init__()
        self.attn1 = AttentionLayer(n_features, hidden_dim, heads)
        self.attn2 = AttentionLayer(hidden_dim, hidden_dim, heads)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(),
                                nn.Dropout(0.3), nn.Linear(32, 1))

    def forward(self, adj, x, mask):
        h = torch.relu(self.attn1(adj, x, mask))
        h = self.drop(h)
        h = torch.relu(self.attn2(adj, h, mask))
        h = h * mask
        emb = h.sum(0) / (mask.sum(0) + 1e-6)
        return torch.sigmoid(self.fc(emb))

# ── MPNN (Message Passing Neural Network with GRU) ──────────────────────────
class MPNNModel(nn.Module):
    def __init__(self, n_features, hidden_dim=128, steps=3):
        super().__init__()
        self.W_msg = nn.Linear(n_features, hidden_dim)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)
        self.steps = steps
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(),
                                nn.Dropout(0.3), nn.Linear(64, 1))
        self.init_h = nn.Linear(n_features, hidden_dim)

    def forward(self, adj, x, mask):
        h = torch.relu(self.init_h(x))
        msg_x = torch.relu(self.W_msg(x))
        for _ in range(self.steps):
            m = adj @ msg_x
            n_real = int(mask.sum().item())
            if n_real > 0:
                h_new = h.clone()
                h_new[:n_real] = self.gru(m[:n_real], h[:n_real])
                h = h_new
            msg_x = h
        h = h * mask
        emb = h.sum(0) / (mask.sum(0) + 1e-6)
        return torch.sigmoid(self.fc(emb))

# ==============================================================================
# STEP 5: Unified training and prediction
# ==============================================================================
MODEL_CONFIGS = {
    'GIN':        {'cls': GINModel,       'epochs': 30, 'lr': 0.001},
    'GraphConv':  {'cls': GraphConvModel,  'epochs': 20, 'lr': 0.001},
    'AttentiveFP':{'cls': AttentiveFPModel,'epochs': 20, 'lr': 0.001},
    'MPNN':       {'cls': MPNNModel,       'epochs': 20, 'lr': 0.001},
}

def train_model(model, graphs_with_labels, device, epochs, lr, labels_all):
    """Train a PyTorch graph model with class weighting."""
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Class weighting
    ys = [y for _, y in graphs_with_labels]
    n_pos = sum(ys)
    n_neg = len(ys) - n_pos
    pos_w = n_neg / max(n_pos, 1)
    weight_map = {0: 1.0, 1: pos_w}
    crit = nn.BCELoss(reduction='none')

    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(graphs_with_labels))
        for idx in perm:
            g, y = graphs_with_labels[idx]
            opt.zero_grad()
            pred = model(g['adj'].to(device), g['feat'].to(device), g['mask'].to(device))
            target = torch.tensor([y], dtype=torch.float32, device=device)
            loss = crit(pred.view(-1), target) * weight_map[y]
            loss.backward()
            opt.step()
    return model

def predict_model(model, smiles_list, indices, graph_cache, device):
    """Predict using a PyTorch graph model."""
    model.eval()
    preds = []
    for i in indices:
        g = graph_cache.get(i)
        if g is None:
            g = smiles_to_graph(smiles_list[i])
            graph_cache[i] = g
        if g is not None:
            with torch.no_grad():
                p = model(g['adj'].to(device), g['feat'].to(device), g['mask'].to(device)).item()
            preds.append(p)
        else:
            preds.append(0.5)
    return np.array(preds)

# ==============================================================================
# STEP 6: MAIN LOOP — Run all 4 models on all tasks/seeds
# ==============================================================================
new_rows = []
fix_checkpoint = f'{RESULTS_DIR}/cell8b_fix_checkpoint.csv'

# Resume support
done_keys_8b = set()
if os.path.exists(fix_checkpoint):
    df_8b = pd.read_csv(fix_checkpoint)
    new_rows = df_8b.to_dict('records')
    done_keys_8b = set(zip(df_8b['Task'], df_8b['Seed'], df_8b['Model']))
    print(f"Resuming Cell 8b: {len(new_rows)} rows already done")

t0 = time.time()

for task_idx, task in enumerate(TOX21_TASKS):
    print(f"\n{'='*60}")
    print(f"[{task_idx+1}/12] {task}")
    print(f"{'='*60}")

    smiles_list, labels = load_tox21_task(df_raw, task)
    train_idx, val_idx, test_idx = scaffold_split(smiles_list)
    y_test = labels[test_idx]

    # Pre-compute graphs (once per task, shared across all models & seeds)
    needed_idx = set(train_idx) | set(test_idx)
    graph_cache = {}
    for i in needed_idx:
        graph_cache[i] = smiles_to_graph(smiles_list[i])
    n_valid = sum(1 for g in graph_cache.values() if g is not None)
    print(f"  Graphs: {n_valid}/{len(needed_idx)} valid")

    for seed in SEEDS:
        row_base = {'Task': task, 'Seed': seed}

        for model_name, cfg in MODEL_CONFIGS.items():
            if (task, seed, model_name) in done_keys_8b:
                continue

            try:
                torch.manual_seed(seed)
                np.random.seed(seed)

                # Prepare training data
                train_data = [(graph_cache[i], labels[i]) for i in train_idx
                              if graph_cache.get(i) is not None]

                # Build and train
                model = cfg['cls'](N_FEAT).to(device)
                model = train_model(model, train_data, device,
                                    cfg['epochs'], cfg['lr'], labels)

                # Predict
                preds = predict_model(model, smiles_list, test_idx, graph_cache, device)
                metrics = compute_all_metrics(y_test, preds)
                new_rows.append({**row_base, 'Model': model_name, **metrics})
                print(f"  {model_name} seed={seed}: ROC={metrics['ROC_AUC']:.3f} ✓")

                del model
                torch.cuda.empty_cache()

            except Exception as e:
                print(f"  {model_name} seed={seed} FAILED: {e}")
                import traceback; traceback.print_exc()
                new_rows.append({**row_base, 'Model': model_name,
                                 'ROC_AUC': np.nan, 'PR_AUC': np.nan,
                                 'MCC': np.nan, 'Balanced_Acc': np.nan})

    # Checkpoint after each task
    pd.DataFrame(new_rows).to_csv(fix_checkpoint, index=False)
    elapsed = (time.time() - t0) / 60
    remaining = elapsed / (task_idx + 1) * (12 - task_idx - 1)
    print(f"  ✓ Checkpoint saved. Elapsed: {elapsed:.0f} min, ~{remaining:.0f} min remaining")

# ==============================================================================
# STEP 7: MERGE with existing good results
# ==============================================================================
print(f"\n{'='*60}")
print("MERGING RESULTS")
print(f"{'='*60}")

df_fixed = pd.DataFrame(new_rows)
df_merged = pd.concat([df_keep, df_fixed], ignore_index=True)

# Save merged results
df_merged.to_csv(f'{RESULTS_DIR}/revised_full_benchmark.csv', index=False)

# Generate summary with mean ± std
summary = df_merged.groupby(['Task', 'Model']).agg(
    ROC_AUC_mean=('ROC_AUC', 'mean'), ROC_AUC_std=('ROC_AUC', 'std'),
    PR_AUC_mean=('PR_AUC', 'mean'),   PR_AUC_std=('PR_AUC', 'std'),
    MCC_mean=('MCC', 'mean'),         MCC_std=('MCC', 'std'),
    BAcc_mean=('Balanced_Acc', 'mean'), BAcc_std=('Balanced_Acc', 'std'),
).reset_index()
summary.to_csv(f'{RESULTS_DIR}/revised_full_benchmark_summary.csv', index=False)

# Verification
print(f"\nTotal rows: {len(df_merged)}")
print(f"Models: {sorted(df_merged['Model'].unique())}")
print(f"Tasks:  {sorted(df_merged['Task'].unique())}")
print(f"Seeds:  {sorted(df_merged['Seed'].unique())}")

expected = len(TOX21_TASKS) * len(SEEDS) * 9  # 9 models
print(f"Expected: {expected}, Got: {len(df_merged)}")

nan_check = df_merged.groupby('Model')['ROC_AUC'].apply(lambda x: x.isna().sum())
print(f"\nRemaining NaNs per model:\n{nan_check}")

# Final pivot table
pivot = df_merged.groupby(['Task','Model'])['ROC_AUC'].mean().reset_index() \
    .pivot(index='Task', columns='Model', values='ROC_AUC')
print(f"\n{pivot.round(3).to_string()}")

elapsed_total = (time.time() - t0) / 60
print(f"\nCell 8b completed in {elapsed_total:.1f} minutes")

Cell 8b: Fix Failed Models — PyTorch on cuda
  GPU: Tesla T4
  VRAM: 15.6 GB
Loaded 360 existing rows
Models: ['ChemBERTa', 'GCN', 'GIN', 'Hybrid_MoE', 'RF', 'XGB']

NaN rows: 0

Will re-run: ['AttentiveFP', 'GraphConv', 'MPNN', 'GIN']
Keeping 300 good rows (RF, XGB, GCN, ChemBERTa, Hybrid_MoE)
Resuming Cell 8b: 120 rows already done

[1/12] NR-AR
  Graphs: 6049/6100 valid
  ✓ Checkpoint saved. Elapsed: 0 min, ~3 min remaining

[2/12] NR-AR-LBD
  Graphs: 5639/5674 valid
  ✓ Checkpoint saved. Elapsed: 1 min, ~3 min remaining

[3/12] NR-AhR
  Graphs: 5457/5499 valid
  ✓ Checkpoint saved. Elapsed: 1 min, ~3 min remaining

[4/12] NR-Aromatase
  Graphs: 4850/4888 valid
  ✓ Checkpoint saved. Elapsed: 1 min, ~2 min remaining

[5/12] NR-ER
  Graphs: 5161/5197 valid
  ✓ Checkpoint saved. Elapsed: 1 min, ~2 min remaining

[6/12] NR-ER-LBD
  Graphs: 5799/5840 valid
  ✓ Checkpoint saved. Elapsed: 2 min, ~2 min remaining

[7/12] NR-PPAR-gamma
  Graphs: 5381/5416 valid
  GIN seed=42: ROC=0.625 ✓
  G

In [ ]:
# =============================================================================
# CELL 9: COMPREHENSIVE ABLATION STUDY (Reviewer Revisions)
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, balanced_accuracy_score
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolDescriptors

# Silence RDKit deprecation warnings
RDLogger.DisableLog('rdApp.*')

# --- GOOGLE DRIVE INTEGRATION ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/tox-report-paper'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CHECKPOINT_PATH = os.path.join(DRIVE_DIR, 'ablation_checkpoints.csv')
    COLAB_ENV = True
except:
    DRIVE_DIR = './results'
    CHECKPOINT_PATH = os.path.join(DRIVE_DIR, 'ablation_checkpoints.csv')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    COLAB_ENV = False

warnings.filterwarnings('ignore')

# --- 1. LOCAL METRICS ---
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    m = {}
    try: m['ROC_AUC'] = roc_auc_score(y_true, y_prob)
    except: m['ROC_AUC'] = np.nan
    try: m['PR_AUC'] = average_precision_score(y_true, y_prob)
    except: m['PR_AUC'] = np.nan
    try: m['MCC'] = matthews_corrcoef(y_true, y_pred)
    except: m['MCC'] = np.nan
    try: m['Balanced_Acc'] = balanced_accuracy_score(y_true, y_pred)
    except: m['Balanced_Acc'] = np.nan
    return m

# --- 2. MULTIPLE REPRESENTATIONS ---
def get_fingerprints_ablation(smiles_list, fp_type='morgan'):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None: mol = Chem.MolFromSmiles('C')
        if fp_type == 'morgan':
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        elif fp_type == 'maccs':
            fp = rdMolDescriptors.GetMACCSKeysFingerprint(mol)
        elif fp_type == 'atompair':
            fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=1024)
        else: fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        fps.append(np.array(fp))
    return np.stack(fps)

# --- 3. EXTENDED META-LEARNER ---
def get_meta_prediction_ablation(expert_probs_test, expert_probs_val=None, y_val=None, variant='lr_none', C=1.0):
    expert_names = sorted(expert_probs_test.keys())
    X_test = np.column_stack([expert_probs_test[n] for n in expert_names])
    if variant == 'averaging': return X_test.mean(axis=1)
    if variant == 'voting': return (X_test > 0.5).astype(float).mean(axis=1)
    X_val = np.column_stack([expert_probs_val[n] for n in expert_names])
    if variant == 'lr_none':
        model = LogisticRegression(C=np.inf, solver='lbfgs', max_iter=1000)
    elif variant == 'lr_l2':
        model = LogisticRegression(penalty='l2', C=C, solver='lbfgs', max_iter=1000)
    model.fit(X_val, y_val)
    return model.predict_proba(X_test)[:, 1]

# --- 4. CONFIGURATION ---
EXPERT_COMBINATIONS = {
    'RF_only': ['RF'], 'XGB_only': ['XGB'], 'GCN_only': ['GCN'],
    'RF+GCN': ['RF', 'GCN'], 'RF+XGB': ['RF', 'XGB'], 'XGB+GCN': ['XGB', 'GCN'],
    'RF+XGB+GCN': ['RF', 'XGB', 'GCN']
}

META_VARIANTS = ['averaging', 'voting', 'lr_none', 'lr_l2']
C_VALUES = [0.1, 1.0, 10.0]
REPRESENTATIONS = ['morgan', 'maccs', 'atompair']

# --- 5. MAIN ABLATION LOOP ---
def run_full_ablation(tasks=TOX21_TASKS, seeds=SEEDS):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    df_raw_data = load_tox21_raw()
    all_results = []

    for task in tasks:
        print(f"\n>>> Starting Task: {task}")
        smiles_list, labels = load_tox21_task(df_raw_data, task)
        train_idx, val_idx, test_idx = scaffold_split(smiles_list)
        y_train, y_val, y_test = labels[train_idx], labels[val_idx], labels[test_idx]

        # Prepare Graphs
        all_graphs = [smiles_to_graph(s) for s in smiles_list]
        vt = [(i, all_graphs[i]) for i in train_idx if all_graphs[i] is not None]
        train_graphs_list = [g for _, g in vt]
        train_labels_list = np.array([labels[i] for i, _ in vt])

        for seed in seeds:
            print(f"  [Seed {seed}] Training experts...")
            torch.manual_seed(seed)
            # GCN Expert
            gcn_m = AdvancedGCN(get_atom_feature_dim()).to(device)
            gcn_m = train_gcn(gcn_m, train_graphs_list, train_labels_list, device)
            gcn_val = predict_gcn(gcn_m, [all_graphs[i] for i in val_idx], device)
            gcn_test = predict_gcn(gcn_m, [all_graphs[i] for i in test_idx], device)

            for rep in REPRESENTATIONS:
                print(f"    Testing Representation: {rep}...")
                X_fp = get_fingerprints_ablation(smiles_list, fp_type=rep)
                X_tr, X_va, X_te = X_fp[train_idx], X_fp[val_idx], X_fp[test_idx]

                # Experts
                rf_m = create_rf(seed); rf_m.fit(X_tr, y_train)
                rf_v, rf_t = rf_m.predict_proba(X_va)[:, 1], rf_m.predict_proba(X_te)[:, 1]

                ratio = float(np.sum(y_train == 0)) / max(np.sum(y_train == 1), 1)
                xgb_m = create_xgb(scale_pos_weight=ratio, seed=seed); xgb_m.fit(X_tr, y_train)
                xgb_v, xgb_t = xgb_m.predict_proba(X_va)[:, 1], xgb_m.predict_proba(X_te)[:, 1]

                experts_v = {'RF': rf_v, 'XGB': xgb_v, 'GCN': gcn_val}
                experts_t = {'RF': rf_t, 'XGB': xgb_t, 'GCN': gcn_test}

                for combo, exp_list in EXPERT_COMBINATIONS.items():
                    if len(exp_list) == 1:
                        metrics = compute_metrics(y_test, experts_t[exp_list[0]])
                        all_results.append({'Task': task, 'Seed': seed, 'Representation': rep if exp_list[0] != 'GCN' else 'Graph', 'Combination': combo, 'Meta_Variant': 'single', 'LR_C': np.nan, **metrics})
                    else:
                        v_dict = {n: experts_v[n] for n in exp_list}
                        t_dict = {n: experts_t[n] for n in exp_list}
                        for variant in META_VARIANTS:
                            c_l = C_VALUES if variant == 'lr_l2' else [np.nan]
                            for c in c_l:
                                probs = get_meta_prediction_ablation(t_dict, expert_probs_val=v_dict, y_val=y_val, variant=variant, C=c)
                                all_results.append({'Task': task, 'Seed': seed, 'Representation': rep, 'Combination': combo, 'Meta_Variant': variant, 'LR_C': c, **compute_metrics(y_test, probs)})

            # Checkpoint
            pd.DataFrame(all_results).to_csv(CHECKPOINT_PATH, index=False)

    final_df = pd.DataFrame(all_results)
    final_df.to_csv(os.path.join(RESULTS_DIR, 'comprehensive_ablation_results.csv'), index=False)
    print("\n>>> Ablation study completed successfully.")
    return final_df

# Run FULL study
display(run_full_ablation().head())


>>> Starting Task: NR-AhR
  [Seed 42] Training experts...
    Testing Representation: morgan...
    Testing Representation: maccs...
    Testing Representation: atompair...
  [Seed 1] Training experts...
    Testing Representation: morgan...
    Testing Representation: maccs...
    Testing Representation: atompair...
  [Seed 7] Training experts...
    Testing Representation: morgan...
    Testing Representation: maccs...
    Testing Representation: atompair...
  [Seed 123] Training experts...
    Testing Representation: morgan...
    Testing Representation: maccs...
    Testing Representation: atompair...
  [Seed 999] Training experts...
    Testing Representation: morgan...
    Testing Representation: maccs...
    Testing Representation: atompair...

>>> Starting Task: NR-AR
  [Seed 42] Training experts...
    Testing Representation: morgan...
    Testing Representation: maccs...
    Testing Representation: atompair...
  [Seed 1] Training experts...
    Testing Representation: morgan

,Task,Seed,Representation,Combination,Meta_Variant,LR_C,ROC_AUC,PR_AUC,MCC,Balanced_Acc
0,NR-AhR,42,morgan,RF_only,single,NaN,0.781290,0.455529,0.236128,0.543471
1,NR-AhR,42,morgan,XGB_only,single,NaN,0.759258,0.423824,0.320970,0.675522
2,NR-AhR,42,Graph,GCN_only,single,NaN,0.788847,0.447485,0.300570,0.635501
3,NR-AhR,42,morgan,RF+GCN,averaging,NaN,0.810136,0.506743,0.393708,0.611135
4,NR-AhR,42,morgan,RF+GCN,voting,NaN,0.667062,0.329859,0.415200,0.666322


In [ ]:
import pandas as pd
import os

# User provided data
data = [
    ['NR-AhR', 42, 'morgan', 'RF_only', 'single', None, 0.781290, 0.455529, 0.236128, 0.543471],
    ['NR-AhR', 42, 'morgan', 'XGB_only', 'single', None, 0.759258, 0.423824, 0.320970, 0.675522],
    ['NR-AhR', 42, 'Graph', 'GCN_only', 'single', None, 0.788847, 0.447485, 0.300570, 0.635501],
    ['NR-AhR', 42, 'morgan', 'RF+GCN', 'averaging', None, 0.810136, 0.506743, 0.393708, 0.611135],
    ['NR-AhR', 42, 'morgan', 'RF+GCN', 'voting', None, 0.667062, 0.329859, 0.415200, 0.666322]
]

columns = ['Task', 'Seed', 'Representation', 'Combination', 'Meta_Variant', 'LR_C', 'ROC_AUC', 'PR_AUC', 'MCC', 'Balanced_Acc']

df_manual = pd.DataFrame(data, columns=columns)

# Drive path
drive_folder = '/content/drive/MyDrive/tox-report-paper'
manual_save_path = os.path.join(drive_folder, 'comprehensive_ablation_results_MANUAL.csv')

try:
    if os.path.exists('/content/drive/MyDrive'):
        os.makedirs(drive_folder, exist_ok=True)
        df_manual.to_csv(manual_save_path, index=False)
        print(f"🎉 Manuel veriler Drive'a başarıyla kaydedildi: {manual_save_path}")
        display(df_manual)
    else:
        print("❌ Hata: Google Drive bağlı değil.")
except Exception as e:
    print(f"❌ Kayıt hatası: {e}")

🎉 Manuel veriler Drive'a başarıyla kaydedildi: /content/drive/MyDrive/tox-report-paper/comprehensive_ablation_results_MANUAL.csv


,Task,Seed,Representation,Combination,Meta_Variant,LR_C,ROC_AUC,PR_AUC,MCC,Balanced_Acc
0,NR-AhR,42,morgan,RF_only,single,None,0.781290,0.455529,0.236128,0.543471
1,NR-AhR,42,morgan,XGB_only,single,None,0.759258,0.423824,0.320970,0.675522
2,NR-AhR,42,Graph,GCN_only,single,None,0.788847,0.447485,0.300570,0.635501
3,NR-AhR,42,morgan,RF+GCN,averaging,None,0.810136,0.506743,0.393708,0.611135
4,NR-AhR,42,morgan,RF+GCN,voting,None,0.667062,0.329859,0.415200,0.666322


In [ ]:
# =============================================================================
# CELL 10: TARGETED CHIRALITY ABLATION STUDY
# =============================================================================
# Purpose:A targeted ablation on 3 representative endpoints.
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, balanced_accuracy_score
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger  # RDKit loglarını kontrol etmek için eklendi
from IPython.display import display

# Standart Python uyarılarını sustur
warnings.filterwarnings('ignore')

# --- RDKIT C++ UYARILARINI (DEPRECATION) TAMAMEN SUSTUR ---
RDLogger.DisableLog('rdApp.*')

# --- GOOGLE DRIVE INTEGRATION ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/tox-report-paper/ablation'
    os.makedirs(DRIVE_DIR, exist_ok=True)
except ImportError:
    DRIVE_DIR = './results/ablation'
    os.makedirs(DRIVE_DIR, exist_ok=True)

OUT_PATH = os.path.join(DRIVE_DIR, 'targeted_chirality_results.csv')

# --- 1. LOCAL METRICS & FINGERPRINT FUNCTIONS ---
def compute_metrics_local(y_true, y_prob, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    try: roc = roc_auc_score(y_true, y_prob)
    except: roc = np.nan
    try: pr = average_precision_score(y_true, y_prob)
    except: pr = np.nan
    try: mcc = matthews_corrcoef(y_true, y_pred)
    except: mcc = np.nan
    try: bacc = balanced_accuracy_score(y_true, y_pred)
    except: bacc = np.nan
    return {'ROC_AUC': roc, 'PR_AUC': pr, 'MCC': mcc, 'Balanced_Acc': bacc}

def get_fingerprints_chiral(smiles_list, use_chirality=False):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None: mol = Chem.MolFromSmiles('C')
        # Bu satır normalde yüzlerce uyarı basar, ama artık RDLogger susturulduğu için tertemiz çalışacak.
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024, useChirality=use_chirality)
        fps.append(np.array(fp))
    return np.stack(fps)

# --- 2. RAPID ABLATION LOOP ---
TARGET_TASKS = ['NR-AhR', 'NR-AR', 'SR-p53']
FIXED_SEED = 42

def run_fast_chirality_ablation():
    print("🚀 Starting Targeted Chirality Ablation (Silent Mode)...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    df_raw = load_tox21_raw()
    all_results = []

    for task in TARGET_TASKS:
        print(f"\n[{task}] Processing...")
        smiles_list, labels = load_tox21_task(df_raw, task)
        train_idx, val_idx, test_idx = scaffold_split(smiles_list)

        y_train_full = labels[train_idx]
        y_test_full = labels[test_idx]

        for include_chirality in [False, True]:
            mode_str = "Chiral_Aware" if include_chirality else "Chiral_Agnostic"
            print(f"  -> Mode: {mode_str}")

            # --- 1. RF (Fingerprints) ---
            X_fp = get_fingerprints_chiral(smiles_list, use_chirality=include_chirality)
            X_train, X_test = X_fp[train_idx], X_fp[test_idx]

            rf = create_rf(FIXED_SEED)
            rf.fit(X_train, y_train_full)
            rf_test = rf.predict_proba(X_test)[:, 1]

            # --- 2. GCN (Graphs) ---
            all_graphs = [smiles_to_graph(s, include_chirality=include_chirality) for s in smiles_list]

            valid_train = [(i, all_graphs[idx]) for i, idx in enumerate(train_idx) if all_graphs[idx] is not None]
            train_graphs_list = [g for _, g in valid_train]
            train_labels_list = np.array([y_train_full[i] for i, _ in valid_train])

            valid_test = [(i, all_graphs[idx]) for i, idx in enumerate(test_idx) if all_graphs[idx] is not None]
            test_graphs_list = [g for _, g in valid_test]
            y_test_filtered = np.array([y_test_full[i] for i, _ in valid_test])

            torch.manual_seed(FIXED_SEED)
            input_dim = get_atom_feature_dim(include_chirality=include_chirality)

            gcn_model = AdvancedGCN(input_dim).to(device)
            gcn_model = train_gcn(gcn_model, train_graphs_list, train_labels_list, device)
            gcn_test = predict_gcn(gcn_model, test_graphs_list, device)

            # --- Record Metrics ---
            rf_metrics = compute_metrics_local(y_test_full, rf_test)
            all_results.append({
                'Task': task, 'Model': 'RF',
                'Chirality_Mode': mode_str, **rf_metrics
            })

            gcn_metrics = compute_metrics_local(y_test_filtered, gcn_test)
            all_results.append({
                'Task': task, 'Model': 'GCN',
                'Chirality_Mode': mode_str, **gcn_metrics
            })

    df_final = pd.DataFrame(all_results)
    df_final.to_csv(OUT_PATH, index=False)
    print(f"\n🎉 Done! Results saved to: {OUT_PATH}")
    display(df_final)
    return df_final

# Kodu Başlat
df_chirality = run_fast_chirality_ablation()

🚀 Starting Targeted Chirality Ablation (Silent Mode)...

[NR-AhR] Processing...
  -> Mode: Chiral_Agnostic
  -> Mode: Chiral_Aware

[NR-AR] Processing...
  -> Mode: Chiral_Agnostic
  -> Mode: Chiral_Aware

[SR-p53] Processing...
  -> Mode: Chiral_Agnostic
  -> Mode: Chiral_Aware

🎉 Done! Results saved to: /content/drive/MyDrive/tox-report-paper/ablation/targeted_chirality_results.csv


,Task,Model,Chirality_Mode,ROC_AUC,PR_AUC,MCC,Balanced_Acc
0,NR-AhR,RF,Chiral_Agnostic,0.781290,0.455529,0.236128,0.543471
1,NR-AhR,GCN,Chiral_Agnostic,0.815946,0.504846,0.389983,0.645102
2,NR-AhR,RF,Chiral_Aware,0.766210,0.436890,0.286855,0.555115
3,NR-AhR,GCN,Chiral_Aware,0.807511,0.478077,0.376523,0.648541
4,NR-AR,RF,Chiral_Agnostic,0.755616,0.460304,0.498464,0.654808
5,NR-AR,GCN,Chiral_Agnostic,0.690020,0.329980,0.411152,0.697189
6,NR-AR,RF,Chiral_Aware,0.743353,0.467561,0.563938,0.671515
7,NR-AR,GCN,Chiral_Aware,0.684089,0.357404,0.000000,0.500000
8,SR-p53,RF,Chiral_Agnostic,0.673128,0.231470,0.155420,0.516592
9,SR-p53,GCN,Chiral_Agnostic,0.649076,0.189468,0.000000,0.500000


In [ ]:
# =============================================================================
# CELL 11: FAIL-SAFE (OVERRIDE) MECHANISM & CASE STUDY ANALYSIS (Reviewer 2)
# =============================================================================
# Purpose: Identify specific molecules where GCN (Deep Learning) fails but
# RF/XGB (Classical ML) intervenes, allowing the Meta-Learner to make the
# correct prediction.
# =============================================================================

import os
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from IPython.display import display

# --- GOOGLE DRIVE INTEGRATION ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    COLAB_ENV = True
    DRIVE_DIR = '/content/drive/MyDrive/tox-report-paper/case_studies'
    os.makedirs(DRIVE_DIR, exist_ok=True)
except ImportError:
    COLAB_ENV = False
    DRIVE_DIR = './results/case_studies'
    os.makedirs(DRIVE_DIR, exist_ok=True)

# --- HELPER FUNCTION (For Fingerprints, in case it's missing from previous cells) ---
def get_morgan_fps(smiles_list):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None: mol = Chem.MolFromSmiles('C')
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        fps.append(np.array(fp))
    return np.stack(fps)

# --- OVERRIDE ANALYSIS FUNCTION ---
def run_override_analysis(task='NR-AhR', seed=42):
    print(f"[{task}] Starting Fail-Safe (Override) Analysis (Seed: {seed})...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Load Data (Using functions defined in previous cells)
    df_raw = load_tox21_raw()
    smiles_list, labels = load_tox21_task(df_raw, task)
    train_idx, val_idx, test_idx = scaffold_split(smiles_list)

    y_train, y_val, y_test = labels[train_idx], labels[val_idx], labels[test_idx]
    smiles_test = [smiles_list[i] for i in test_idx]

    # 2. Extract Features for Models
    X_fp = get_morgan_fps(smiles_list)
    X_train_fp, X_val_fp, X_test_fp = X_fp[train_idx], X_fp[val_idx], X_fp[test_idx]

    all_graphs = [smiles_to_graph(s) for s in smiles_list]
    valid_train = [(i, all_graphs[i]) for i in train_idx if all_graphs[i] is not None]
    train_graphs = [g for _, g in valid_train]
    train_labels = np.array([labels[i] for i, _ in valid_train])
    val_graphs = [all_graphs[i] for i in val_idx]
    test_graphs = [all_graphs[i] for i in test_idx]

    # 3. RF Training and Predictions
    print("  -> Training Random Forest...")
    rf = create_rf(seed)
    rf.fit(X_train_fp, y_train)
    rf_val = rf.predict_proba(X_val_fp)[:, 1]
    rf_test = rf.predict_proba(X_test_fp)[:, 1]

    # 4. XGB Training and Predictions
    print("  -> Training XGBoost...")
    ratio = float(np.sum(y_train == 0)) / max(np.sum(y_train == 1), 1)
    xgb = create_xgb(scale_pos_weight=ratio, seed=seed)
    xgb.fit(X_train_fp, y_train)
    xgb_val = xgb.predict_proba(X_val_fp)[:, 1]
    xgb_test = xgb.predict_proba(X_test_fp)[:, 1]

    # 5. GCN Training and Predictions
    print("  -> Training GCN...")
    torch.manual_seed(seed)
    gcn_model = AdvancedGCN(get_atom_feature_dim()).to(device)
    gcn_model = train_gcn(gcn_model, train_graphs, train_labels, device)
    gcn_val = predict_gcn(gcn_model, val_graphs, device)
    gcn_test = predict_gcn(gcn_model, test_graphs, device)

    # 6. Meta-Learner (Logistic Regression) Training
    print("  -> Training Meta-Learner (Hybrid MoE) and analyzing weights...")
    X_meta_val = np.column_stack([rf_val, xgb_val, gcn_val])
    X_meta_test = np.column_stack([rf_test, xgb_test, gcn_test])

    meta_model = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs')
    meta_model.fit(X_meta_val, y_val)
    hybrid_test = meta_model.predict_proba(X_meta_test)[:, 1]

    # Print Meta-Learner Weights (Crucial for discussion in the manuscript)
    weights = meta_model.coef_[0]
    print("\n" + "="*50)
    print("🧠 META-LEARNER LEARNED WEIGHTS:")
    print(f"  RF Weight  : {weights[0]:.4f}")
    print(f"  XGB Weight : {weights[1]:.4f}")
    print(f"  GCN Weight : {weights[2]:.4f}")
    print("="*50 + "\n")

    # 7. Convert Results to DataFrame
    df_results = pd.DataFrame({
        'SMILES': smiles_test,
        'True_Label': y_test,
        'RF_Prob': rf_test,
        'XGB_Prob': xgb_test,
        'GCN_Prob': gcn_test,
        'Hybrid_Prob': hybrid_test
    })

    # 8. FIND 'FAIL-SAFE' CASES (Override Analysis)
    # Scenario 1: GCN incorrectly predicts the molecule is non-toxic (0) but it is actually toxic (1).
    # RF/XGB recognizes the toxicity, and the Hybrid model correctly classifies it (1).
    fail_safe_cases = df_results[
        (df_results['True_Label'] == 1) &
        (df_results['GCN_Prob'] < 0.5) &      # GCN failed (False Negative)
        (df_results['Hybrid_Prob'] >= 0.5)    # Hybrid succeeded (True Positive)
    ].copy()

    fail_safe_cases['GCN_Error_Margin'] = 0.5 - fail_safe_cases['GCN_Prob']
    fail_safe_cases = fail_safe_cases.sort_values(by='GCN_Error_Margin', ascending=False)

    csv_path = os.path.join(DRIVE_DIR, f'override_analysis_{task}_seed{seed}.csv')
    fail_safe_cases.to_csv(csv_path, index=False)

    print(f"✅ Found {len(fail_safe_cases)} molecules where GCN failed but the Hybrid model succeeded!")
    print(f"💾 Cases saved to: {csv_path}\n")

    # Draw the top 3 cases on screen
    if len(fail_safe_cases) > 0:
        top_cases = fail_safe_cases.head(3)
        print("🔬 TOP 3 CRITICAL 'FAIL-SAFE' CASES (Visualization):")
        display(top_cases[['SMILES', 'True_Label', 'GCN_Prob', 'RF_Prob', 'Hybrid_Prob']])

        mols = [Chem.MolFromSmiles(s) for s in top_cases['SMILES']]
        legends = [f"GCN: {row['GCN_Prob']:.2f} | RF: {row['RF_Prob']:.2f} | Hyb: {row['Hybrid_Prob']:.2f}"
                   for _, row in top_cases.iterrows()]

        img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), legends=legends)
        display(img)

        # Save the image to Drive (Can be used for Figure 4 in the manuscript)
        img.save(os.path.join(DRIVE_DIR, f'Figure4_FailSafe_Cases_{task}.png'))
        print(f"🖼️ Molecule drawings saved to Drive.")

    return df_results, fail_safe_cases

# Run the test
df_all, df_failsafe = run_override_analysis(task='NR-AhR', seed=42)

In [ ]:
# =============================================================================
# CELL 12: STATISTICAL EQUIVALENCE & EFFECT SIZE ANALYSIS (Reviewer 2)
# =============================================================================
# Purpose: Address the reviewer's concern regarding the misuse of the Wilcoxon
# test for claiming "equivalence". This cell implements:
# 1. Cohen's d (Effect Size) with 95% Confidence Intervals
# 2. TOST (Two One-Sided Tests) for formal statistical equivalence
# =============================================================================

import os
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- GOOGLE DRIVE INTEGRATION ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    COLAB_ENV = True
    DRIVE_DIR = '/content/drive/MyDrive/tox-report-paper/statistics'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CSV_PATH = '/content/drive/MyDrive/tox-report-paper/revised_full_benchmark.csv'
except ImportError:
    COLAB_ENV = False
    DRIVE_DIR = './results/statistics'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CSV_PATH = './revised_full_benchmark.csv' # Adjust path if running locally

# --- STATISTICAL FUNCTIONS ---

def calculate_cohens_d(group1, group2):
    """Calculates paired Cohen's d effect size."""
    diff = np.array(group1) - np.array(group2)
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    if std_diff == 0:
        return 0.0
    return mean_diff / std_diff

def calculate_confidence_interval(group1, group2, alpha=0.05):
    """Calculates 95% Confidence Interval for the mean difference."""
    diff = np.array(group1) - np.array(group2)
    mean_diff = np.mean(diff)
    se = stats.sem(diff)
    degrees_of_freedom = len(diff) - 1
    t_stat = stats.t.ppf(1 - alpha/2, degrees_of_freedom)
    margin_of_error = t_stat * se
    return mean_diff - margin_of_error, mean_diff + margin_of_error

def tost_equivalence(group1, group2, margin=0.05):
    """
    Performs Two One-Sided Tests (TOST) for equivalence.
    Null Hypothesis: The difference is outside the equivalence margin [-margin, margin].
    Alternative Hypothesis: The difference is strictly within the margin (Equivalence).
    """
    diff = np.array(group1) - np.array(group2)
    mean_diff = np.mean(diff)
    se = stats.sem(diff)
    degrees_of_freedom = len(diff) - 1

    # Test 1: Mean difference > -margin
    t1 = (mean_diff - (-margin)) / se
    p1 = 1 - stats.t.cdf(t1, degrees_of_freedom)

    # Test 2: Mean difference < margin
    t2 = (mean_diff - margin) / se
    p2 = stats.t.cdf(t2, degrees_of_freedom)

    # The TOST p-value is the maximum of the two p-values
    p_tost = max(p1, p2)
    is_equivalent = p_tost < 0.05

    return p_tost, is_equivalent

# --- MAIN ANALYSIS SCRIPT ---
def run_statistical_analysis():
    print("🔬 Starting Advanced Statistical Analysis (TOST & Effect Sizes)...\n")

    # 1. Load the benchmark data
    if not os.path.exists(CSV_PATH):
        # Fallback to local filename if Drive path fails
        fallback_path = 'revised_full_benchmark.csv'
        if os.path.exists(fallback_path):
            df = pd.read_csv(fallback_path)
        else:
            print(f"❌ ERROR: Cannot find {CSV_PATH}. Please ensure the benchmark CSV is uploaded/available.")
            return
    else:
        df = pd.read_csv(CSV_PATH)

    print(f"✅ Data loaded successfully. Total records: {len(df)}")

    # 2. Aggregate data: Mean over seeds for each Task and Model
    # We focus on ROC_AUC and PR_AUC
    agg_df = df.groupby(['Task', 'Model'])[['ROC_AUC', 'PR_AUC']].mean().reset_index()

    # 3. Define the comparisons to make
    target_model = 'Hybrid_MoE'
    baselines = ['GraphConv', 'GCN', 'AttentiveFP', 'RF'] # Models to compare against
    metrics = ['ROC_AUC', 'PR_AUC']
    equivalence_margin = 0.05 # 5% margin is standard for ML equivalence

    results = []

    for metric in metrics:
        print(f"\n" + "="*50)
        print(f"📊 ANALYZING METRIC: {metric}")
        print("="*50)

        target_scores = agg_df[agg_df['Model'] == target_model].sort_values('Task')[metric].values

        for base in baselines:
            if base not in agg_df['Model'].values:
                continue

            base_scores = agg_df[agg_df['Model'] == base].sort_values('Task')[metric].values

            # Ensure we are comparing the exact same tasks
            if len(target_scores) != len(base_scores):
                print(f"⚠️ Warning: Task count mismatch between {target_model} and {base}. Skipping.")
                continue

            # Wilcoxon Signed-Rank Test (Standard Difference)
            stat, p_wilcoxon = stats.wilcoxon(target_scores, base_scores)

            # Effect Size (Cohen's d)
            cohens_d = calculate_cohens_d(target_scores, base_scores)

            # 95% Confidence Interval
            ci_lower, ci_upper = calculate_confidence_interval(target_scores, base_scores)

            # TOST Equivalence Test
            p_tost, is_equivalent = tost_equivalence(target_scores, base_scores, margin=equivalence_margin)

            mean_diff = np.mean(target_scores - base_scores)

            # Interpret the Effect Size
            if abs(cohens_d) < 0.2: d_interpretation = "Negligible"
            elif abs(cohens_d) < 0.5: d_interpretation = "Small"
            elif abs(cohens_d) < 0.8: d_interpretation = "Medium"
            else: d_interpretation = "Large"

            results.append({
                'Metric': metric,
                'Comparison': f"{target_model} vs {base}",
                'Mean_Diff': mean_diff,
                'CI_95%_Lower': ci_lower,
                'CI_95%_Upper': ci_upper,
                'Wilcoxon_p_val': p_wilcoxon,
                'TOST_p_val': p_tost,
                'Is_Equivalent (Margin=5%)': is_equivalent,
                'Cohens_d': cohens_d,
                'Effect_Size': d_interpretation
            })

            print(f"\n--- {target_model} vs {base} ---")
            print(f"  Mean Difference : {mean_diff:+.4f} (95% CI: [{ci_lower:+.4f}, {ci_upper:+.4f}])")
            print(f"  Wilcoxon p-value: {p_wilcoxon:.4f} " + ("(Significant Diff)" if p_wilcoxon < 0.05 else "(No Significant Diff)"))
            print(f"  Cohen's d       : {cohens_d:+.4f} ({d_interpretation} Effect)")
            print(f"  TOST p-value    : {p_tost:.4f} " + ("(EQUIVALENT ✅)" if is_equivalent else "(NOT EQUIVALENT ❌)"))

    # 4. Save results
    results_df = pd.DataFrame(results)
    out_file = os.path.join(DRIVE_DIR, 'statistical_tests_TOST_Cohens_d.csv')
    results_df.to_csv(out_file, index=False)
    print(f"\n💾 Statistical results successfully saved to: {out_file}")

    return results_df

# Execute the analysis
df_stats = run_statistical_analysis()

In [ ]:
# =============================================================================
# CELL 12-2 NEW: STATISTICAL EQUIVALENCE & EFFECT SIZE ANALYSIS (All Modern Baselines)
# =============================================================================
# Purpose: Address the reviewer's concern regarding the misuse of the Wilcoxon
# test for claiming "equivalence". Compares the Hybrid_MoE against all modern
# DL architectures (AttentiveFP, ChemBERTa, GIN, MPNN) and classical ones.
# =============================================================================

import os
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- GOOGLE DRIVE INTEGRATION ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    COLAB_ENV = True
    DRIVE_DIR = '/content/drive/MyDrive/tox-report-paper/statistics'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CSV_PATH = '/content/drive/MyDrive/tox-report-paper/revised_full_benchmark.csv'
except ImportError:
    COLAB_ENV = False
    DRIVE_DIR = './results/statistics'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CSV_PATH = './revised_full_benchmark.csv'

# --- STATISTICAL FUNCTIONS ---

def calculate_cohens_d(group1, group2):
    diff = np.array(group1) - np.array(group2)
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    if std_diff == 0:
        return 0.0
    return mean_diff / std_diff

def calculate_confidence_interval(group1, group2, alpha=0.05):
    diff = np.array(group1) - np.array(group2)
    mean_diff = np.mean(diff)
    se = stats.sem(diff)
    degrees_of_freedom = len(diff) - 1
    t_stat = stats.t.ppf(1 - alpha/2, degrees_of_freedom)
    margin_of_error = t_stat * se
    return mean_diff - margin_of_error, mean_diff + margin_of_error

def tost_equivalence(group1, group2, margin=0.05):
    diff = np.array(group1) - np.array(group2)
    mean_diff = np.mean(diff)
    se = stats.sem(diff)
    degrees_of_freedom = len(diff) - 1

    t1 = (mean_diff - (-margin)) / se
    p1 = 1 - stats.t.cdf(t1, degrees_of_freedom)

    t2 = (mean_diff - margin) / se
    p2 = stats.t.cdf(t2, degrees_of_freedom)

    p_tost = max(p1, p2)
    is_equivalent = p_tost < 0.05
    return p_tost, is_equivalent

# --- MAIN ANALYSIS SCRIPT ---
def run_statistical_analysis():
    print("🔬 Starting Advanced Statistical Analysis (TOST & Effect Sizes)...\n")

    if not os.path.exists(CSV_PATH):
        fallback_path = 'revised_full_benchmark.csv'
        if os.path.exists(fallback_path):
            df = pd.read_csv(fallback_path)
        else:
            print(f"❌ ERROR: Cannot find CSV. Please ensure the benchmark CSV is uploaded/available.")
            return
    else:
        df = pd.read_csv(CSV_PATH)

    print(f"✅ Data loaded successfully. Total records: {len(df)}")

    # Average across seeds
    agg_df = df.groupby(['Task', 'Model'])[['ROC_AUC', 'PR_AUC']].mean().reset_index()

    # THE FULL LIST OF MODERN DL & CLASSICAL BASELINES
    target_model = 'Hybrid_MoE'
    baselines = ['GraphConv', 'GCN', 'AttentiveFP', 'ChemBERTa', 'GIN', 'MPNN', 'RF', 'XGB']
    metrics = ['ROC_AUC', 'PR_AUC']
    equivalence_margin = 0.05

    results = []

    for metric in metrics:
        print(f"\n" + "="*60)
        print(f"📊 ANALYZING METRIC: {metric}")
        print("="*60)

        target_scores = agg_df[agg_df['Model'] == target_model].sort_values('Task')[metric].values

        for base in baselines:
            if base not in agg_df['Model'].values:
                continue

            base_scores = agg_df[agg_df['Model'] == base].sort_values('Task')[metric].values

            if len(target_scores) != len(base_scores):
                continue

            stat, p_wilcoxon = stats.wilcoxon(target_scores, base_scores)
            cohens_d = calculate_cohens_d(target_scores, base_scores)
            ci_lower, ci_upper = calculate_confidence_interval(target_scores, base_scores)
            p_tost, is_equivalent = tost_equivalence(target_scores, base_scores, margin=equivalence_margin)

            mean_diff = np.mean(target_scores - base_scores)

            if abs(cohens_d) < 0.2: d_interpretation = "Negligible"
            elif abs(cohens_d) < 0.5: d_interpretation = "Small"
            elif abs(cohens_d) < 0.8: d_interpretation = "Medium"
            else: d_interpretation = "Large"

            results.append({
                'Metric': metric,
                'Comparison': f"{target_model} vs {base}",
                'Mean_Diff': mean_diff,
                'CI_95%_Lower': ci_lower,
                'CI_95%_Upper': ci_upper,
                'Wilcoxon_p_val': p_wilcoxon,
                'TOST_p_val': p_tost,
                'Is_Equivalent': is_equivalent,
                'Cohens_d': cohens_d,
                'Effect_Size': d_interpretation
            })

            print(f"\n--- {target_model} vs {base} ---")
            print(f"  Mean Difference : {mean_diff:+.4f} (95% CI: [{ci_lower:+.4f}, {ci_upper:+.4f}])")
            print(f"  Wilcoxon p-value: {p_wilcoxon:.4f} " + ("(Significant Diff)" if p_wilcoxon < 0.05 else "(No Significant Diff)"))
            print(f"  Cohen's d       : {cohens_d:+.4f} ({d_interpretation} Effect)")
            print(f"  TOST p-value    : {p_tost:.4f} " + ("(EQUIVALENT ✅)" if is_equivalent else "(NOT EQUIVALENT ❌)"))

    results_df = pd.DataFrame(results)
    out_file = os.path.join(DRIVE_DIR, 'statistical_tests_TOST_Cohens_d_FULL.csv')
    results_df.to_csv(out_file, index=False)
    print(f"\n💾 Statistical results successfully saved to: {out_file}")

    return results_df

# Execute the analysis
df_stats = run_statistical_analysis()

In [ ]:
# =============================================================================
# CELL 13: DATASET STATISTICS & OOD SPLIT ANALYSIS (Minor Revisions)
# =============================================================================
# Purpose: Address Reviewer 2's minor comments regarding:
# 1. Per-endpoint dataset statistics (positives, missing labels, etc.)
# 2. Scaffold splitting OOD (Out-of-Distribution) distance using Tanimoto
# =============================================================================

import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
from IPython.display import display

# --- GOOGLE DRIVE INTEGRATION ---
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/tox-report-paper/statistics'
    os.makedirs(DRIVE_DIR, exist_ok=True)
except ImportError:
    DRIVE_DIR = './results/statistics'
    os.makedirs(DRIVE_DIR, exist_ok=True)

# --- HELPER FUNCTION: FINGERPRINTS ---
def get_morgan_fps_rdkit(smiles_list):
    """Returns RDKit ExplicitBitVect objects for extremely fast Tanimoto calculation."""
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None: mol = Chem.MolFromSmiles('C')
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        fps.append(fp)
    return fps

# --- MAIN ANALYSIS SCRIPT ---
def run_minor_revisions_analysis(tasks=TOX21_TASKS):
    print("📊 Starting Dataset Statistics & OOD Split Analysis...\n")

    # Load raw data using your existing function
    df_raw = load_tox21_raw()
    total_molecules_in_dataset = len(df_raw)

    stats_results = []

    for task in tasks:
        print(f"  -> Processing Task: {task}")

        # ==========================================
        # 1. DATASET STATISTICS (Missing labels, etc.)
        # ==========================================
        task_data = df_raw[df_raw[task].notnull()]
        valid_count = len(task_data)
        missing_count = total_molecules_in_dataset - valid_count

        positives = int(task_data[task].sum())
        negatives = valid_count - positives
        positive_rate = (positives / valid_count) * 100

        # ==========================================
        # 2. OOD SPLIT ANALYSIS (Tanimoto Distance)
        # ==========================================
        smiles_list, labels = load_tox21_task(df_raw, task)
        train_idx, val_idx, test_idx = scaffold_split(smiles_list)

        train_smiles = [smiles_list[i] for i in train_idx]
        test_smiles = [smiles_list[i] for i in test_idx]

        train_fps = get_morgan_fps_rdkit(train_smiles)
        test_fps = get_morgan_fps_rdkit(test_smiles)

        # Calculate max Tanimoto similarity for each test molecule against ALL train molecules
        test_similarities = []
        for test_fp in test_fps:
            # BulkTanimotoSimilarity is highly optimized in C++
            sims = DataStructs.BulkTanimotoSimilarity(test_fp, train_fps)
            test_similarities.append(max(sims))

        avg_max_similarity = np.mean(test_similarities)
        avg_ood_distance = 1.0 - avg_max_similarity # Distance = 1 - Similarity

        stats_results.append({
            'Endpoint': task,
            'Valid_Labels': valid_count,
            'Missing_Labels': missing_count,
            'Positives (Active)': positives,
            'Negatives (Inactive)': negatives,
            'Positive_Rate (%)': round(positive_rate, 2),
            'Test_Size': len(test_idx),
            'Avg_Max_Tanimoto_Sim': round(avg_max_similarity, 4),
            'Avg_OOD_Distance': round(avg_ood_distance, 4)
        })

    df_stats = pd.DataFrame(stats_results)

    # Save to Drive
    out_path = os.path.join(DRIVE_DIR, 'dataset_and_ood_statistics.csv')
    df_stats.to_csv(out_path, index=False)

    print("\n✅ Analysis Complete!")
    print(f"💾 Results saved to: {out_path}\n")

    return df_stats

# Execute
df_minor_stats = run_minor_revisions_analysis()
display(df_minor_stats)